In [1]:
import os, sys, pickle
import jax, jaxlib
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scipy
from scipy.io import loadmat
import numpy as np
import h5py
from pathlib import Path

import torch
import flax
from flax import linen as nn
import optax
from sklearn.model_selection import train_test_split
from typing import Callable, Sequence

from tqdm import tqdm
from utils_jax import *

In [2]:
# seed = 42
seed = 21634
np.random.seed(seed)
key = jax.random.PRNGKey(seed)
print(f"Seed set with value = {seed}")

Seed set with value = 21634


In [3]:
#Combining all .mat files into one
base_path = "/home/dnayak2/data_sgoswam4/Dibya/Datasets/2D_rotation_advection/Case3_quarter_rotation_coarsen2/samples/"
print(base_path)

dataset_lst = []

for i in range(len([f for f in os.listdir(base_path) if os.path.isfile(os.path.join(base_path, f))])):
    data = loadmat(os.path.join(base_path, f"sample_{i+1:04d}.mat"))
    dataset_lst.append(data['snapshots'])
    
    if i==0:
        X = jnp.array(data['x'])
        Y = jnp.array(data['y'])
        tspan = jnp.array(data['t_vec'])
        dt = jnp.array(data['dt'])
        dt_saved = jnp.array(data['dt_saved'])
        
    del data

/home/dnayak2/data_sgoswam4/Dibya/Datasets/2D_rotation_advection/Case3_quarter_rotation_coarsen2/samples/


In [4]:
outputs = jnp.array(dataset_lst)
#Only consider first 101 (0:100) timesteps
outputs = outputs[:,:101,:,:]

Ns, Nt, Nx, Ny = outputs.shape
print(f"Ns: {Ns}, Nt: {Nt}, Nx: {Nx}, Ny: {Ny}")

#Get the xspan and yspan
xspan = jnp.unique(X)
yspan = jnp.unique(Y)

#Print the xspan and yspan
print(f"xspan: {xspan}, {xspan.shape}")
print("\n")
print(f"yspan: {yspan}, {yspan.shape}")
print("\n")

#Print time vector
tspan = (tspan[:,:Nt]).T
print(f"tspan: {tspan}, {tspan.shape}")
print("\n")
print(f"Actual dt: {dt}, Saved dt: {dt_saved}")

Ns: 800, Nt: 101, Nx: 65, Ny: 65
xspan: [0.       0.015625 0.03125  0.046875 0.0625   0.078125 0.09375  0.109375
 0.125    0.140625 0.15625  0.171875 0.1875   0.203125 0.21875  0.234375
 0.25     0.265625 0.28125  0.296875 0.3125   0.328125 0.34375  0.359375
 0.375    0.390625 0.40625  0.421875 0.4375   0.453125 0.46875  0.484375
 0.5      0.515625 0.53125  0.546875 0.5625   0.578125 0.59375  0.609375
 0.625    0.640625 0.65625  0.671875 0.6875   0.703125 0.71875  0.734375
 0.75     0.765625 0.78125  0.796875 0.8125   0.828125 0.84375  0.859375
 0.875    0.890625 0.90625  0.921875 0.9375   0.953125 0.96875  0.984375
 1.      ], (65,)


yspan: [0.       0.015625 0.03125  0.046875 0.0625   0.078125 0.09375  0.109375
 0.125    0.140625 0.15625  0.171875 0.1875   0.203125 0.21875  0.234375
 0.25     0.265625 0.28125  0.296875 0.3125   0.328125 0.34375  0.359375
 0.375    0.390625 0.40625  0.421875 0.4375   0.453125 0.46875  0.484375
 0.5      0.515625 0.53125  0.546875 0.5625   0.578125 0.

In [5]:
#Use dt_saved as dt_val
dt_val = dt_saved.squeeze()
dt_val

Array(0.00673435, dtype=float32)

In [6]:
tt = 40
print(f"End timestep of training: {tt}")

#Creating the input and output training data
init_timestep = 0
end_timestep = tt

input_data_NN = outputs[:, init_timestep, :, :]    
output_data_NN = outputs[:, init_timestep+1, :, :]

for i in range(init_timestep+1, end_timestep):
    input_data_NN = jnp.vstack((input_data_NN, outputs[:,i,:,:]))
    output_data_NN = jnp.vstack((output_data_NN, outputs[:,i+1,:,:]))
print(input_data_NN.shape, output_data_NN.shape)


#Reshaping the output_data_NN from (ns*nt//2, nx, ny) to (ns*nt//2, nx*ny)
#Input_data_NN remains as it is, i.e., (ns*nt//2, nx, ny)
output_data_NN = output_data_NN.reshape(output_data_NN.shape[0], 
                                        output_data_NN.shape[1]*output_data_NN.shape[2])
print(input_data_NN.shape, output_data_NN.shape)


input_data_NN_train, input_data_NN_test, output_data_NN_train, output_data_NN_test = \
                        train_test_split(input_data_NN, output_data_NN, test_size = 0.2, random_state = 42)
print(input_data_NN_train.shape, input_data_NN_test.shape, 
      output_data_NN_train.shape, output_data_NN_test.shape)


#Freeing memory by deleting input_data_NN and output_data_NN
del input_data_NN, output_data_NN

End timestep of training: 40
(32000, 65, 65) (32000, 65, 65)
(32000, 65, 65) (32000, 4225)
(25600, 65, 65) (6400, 65, 65) (25600, 4225) (6400, 4225)


In [7]:
class branch_net(nn.Module):

    layer_sizes: Sequence[int] 
    activation: Callable
    
    @nn.compact
    def __call__(self, x):
        init = nn.initializers.glorot_normal()
        
        # #x has shape (ns, nx, ny) - so add channel dimension: (ns, nx, ny, nc)
        x = x[..., jnp.newaxis]
        
        #Convolutional layers
        x = nn.Conv(features = 64, kernel_size = (3,3), strides = 1, padding = "SAME")(x)
        x = nn.gelu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides = (2, 2), padding = "SAME")

        x = nn.Conv(features = 64, kernel_size = (3,3), strides = 1, padding = "SAME")(x)
        x = nn.gelu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides = (2, 2), padding = "SAME")
        
        x = nn.Conv(features = 64, kernel_size = (2, 2), strides = 1, padding = "SAME")(x)
        x = nn.gelu(x)
        x = nn.avg_pool(x, window_shape = (2,2), strides = (2,2), padding = "SAME")
        
        x = x.flatten()   #flatten
        
        #MLP layers
        for i, layer in enumerate(self.layer_sizes[:-1]):
            x = nn.Dense(layer, kernel_init = init)(x)
            x = self.activation(x)
        x = nn.Dense(self.layer_sizes[-1], kernel_init = init)(x)
        return x


class trunk_net(nn.Module):
    trunk_layer_config: Sequence[int]
    activation: Callable
    
    @nn.compact
    def __call__(self, x):
        
        init = nn.initializers.glorot_normal()
        
        #Trunk network forward pass
        for i, layer_size in enumerate(self.trunk_layer_config):
            x = nn.Dense(layer_size, kernel_init = init)(x)
            x = self.activation(x)
        return x

def add_fourier_features(inputs, num_frequencies=10, max_freq=10):
    x = inputs[:, 0:1]
    y = inputs[:, 1:2]
    
    freqs = jnp.pi * jnp.linspace(1, max_freq, num_frequencies).reshape(1, -1)
    
    x_feat = jnp.concatenate([jnp.sin(freqs * x), jnp.cos(freqs * x)], axis=-1)
    y_feat = jnp.concatenate([jnp.sin(freqs * y), jnp.cos(freqs * y)], axis=-1)
    
    return jnp.concatenate([inputs, x_feat, y_feat], axis=-1)

class WaveletAct(nn.Module):
    init_w1: float = 1.0
    init_w2: float = 1.0
    init_omega: float = 1.0

    @nn.compact
    def __call__(self, x):
        w1 = self.param('w1', nn.initializers.constant(self.init_w1), (1,))
        w2 = self.param('w2', nn.initializers.constant(self.init_w2), (1,))
        omega = self.param('omega', nn.initializers.constant(self.init_omega), (1,))
        return w1 * jnp.sin(omega * x) + w2 * jnp.cos(omega * x)

class DeepONet(nn.Module):

    branch_net_config: Sequence[int]
    trunk_net_config: Sequence[int]
    use_Fourier_feat: bool = True

    def setup(self):
        self.branch_net = branch_net(self.branch_net_config, nn.gelu)
        self.trunk_net = trunk_net(self.trunk_net_config, WaveletAct())

    @nn.compact
    def __call__(self, x_branch, x_trunk):

        if self.use_Fourier_feat:
            #Encode x_trunk into fourier features
            x_trunk = add_fourier_features(x_trunk)
        
        #Vectorize over multiple samples of input functions
        branch_outputs = jax.vmap(self.branch_net, in_axes = 0)(x_branch)
        
        #Vectorize over multiple query points
        trunk_outputs = jax.vmap(self.trunk_net, in_axes = 0)(x_trunk)
        bias = self.param('bias', nn.initializers.zeros, (1,))      
        
        inner_product = jnp.einsum('ik,jk->ij', branch_outputs, trunk_outputs)
        inner_product+=bias
        return inner_product

In [8]:
#Form branch and trunk inputs train

#Create for trunk network
[x,y] = jnp.meshgrid(xspan, yspan, indexing = 'ij')
grid = jnp.transpose(jnp.array([x.flatten(), y.flatten()]))
print(grid.shape)
print(grid)


#Creating the training data for branch and trunk inputs
branch_inputs_train = input_data_NN_train
trunk_inputs_train = grid
outputs_train = output_data_NN_train

print("Shape of branch inputs train: ",branch_inputs_train.shape)
print("Shape of trunk inputs train: ",trunk_inputs_train.shape)
print("Shape of outputs train: ",outputs_train.shape)
print("Shape of grid: ",grid.shape)


#For branch and trunk inputs test
branch_inputs_test = input_data_NN_test
trunk_inputs_test = grid
outputs_test = output_data_NN_test

print("Shape of branch inputs test: ",branch_inputs_test.shape)
print("Shape of trunk inputs test: ",trunk_inputs_test.shape)
print("Shape of outputs test: ",outputs_test.shape)
print("Shape of grid: ",grid.shape)


#DeepONet settings
num_sensor_locations = branch_inputs_train.shape[1]
num_query_locations = 2
latent_vector_size = 100

branch_network_layer_sizes = [256, 128] + [latent_vector_size]
trunk_network_layer_sizes = [128]*6 + [latent_vector_size]

model = DeepONet(branch_network_layer_sizes, trunk_network_layer_sizes)
model_fn = jax.jit(model.apply)
print(f"Model: {model}")

(4225, 2)
[[0.       0.      ]
 [0.       0.015625]
 [0.       0.03125 ]
 ...
 [1.       0.96875 ]
 [1.       0.984375]
 [1.       1.      ]]
Shape of branch inputs train:  (25600, 65, 65)
Shape of trunk inputs train:  (4225, 2)
Shape of outputs train:  (25600, 4225)
Shape of grid:  (4225, 2)
Shape of branch inputs test:  (6400, 65, 65)
Shape of trunk inputs test:  (4225, 2)
Shape of outputs test:  (6400, 4225)
Shape of grid:  (4225, 2)
Model: DeepONet(
    # attributes
    branch_net_config = [256, 128, 100]
    trunk_net_config = [128, 128, 128, 128, 128, 128, 100]
    use_Fourier_feat = True
)


In [9]:
# Initialize model parameters
key = jax.random.PRNGKey(seed)   #42
params = model.init(key, branch_inputs_train[0:1, ...], trunk_inputs_train[0:1, ...])

# # Optimizer setup
lr_scheduler = optax.schedules.exponential_decay(init_value=1e-3, transition_steps=5000, decay_rate=0.95)
optimizer = optax.adam(learning_rate=lr_scheduler)
opt_state = optimizer.init(params)

training_loss_history = []
test_loss_history = []
num_epochs = int(1.5e5)
batch_size = 64

min_test_loss = jnp.inf

filepath = Path(f'DeepONet_NODE/TI_DON_16767661_seed21634', parents=True, exist_ok=True)
filename = f"model_params_best_{seed}.pkl"

In [10]:
# @jax.jit
def inference_ab(u_curr, u_prev, trunk_inputs_test, dt):
    
    # Step 1: Apply the predictor (Adams-Bashforth) using u_curr and u_prev
    u_dot_curr = model_fn(best_params, u_curr, trunk_inputs_test)  # Predict the rate of change at u_curr
    u_dot_prev = model_fn(best_params, u_prev, trunk_inputs_test)  # Predict the rate of change at u_prev
    
    #Reshaping u_dot_curr and u_dot_prev to broadcast compatible with u_curr
    u_dot_curr = u_dot_curr.reshape(u_dot_curr.shape[0], Nx, Ny)
    u_dot_prev = u_dot_prev.reshape(u_dot_prev.shape[0], Nx, Ny)
    
    # Adams-Bashforth predictor (using previous two points)
    u_pred = u_curr + dt * (1.5 * u_dot_curr - 0.5 * u_dot_prev)
    
    # Step 2: Apply the corrector (Adams-Moulton) using the predicted u_pred
    u_dot_pred = model_fn(best_params, u_pred, trunk_inputs_test)  # Predict the rate of change at u_pred
    
    #Reshaping u_dot_pred to broadcast compatible with u_curr, u_dot_curr, u_dot_prev
    u_dot_pred = u_dot_pred.reshape(u_dot_pred.shape[0], Nx, Ny)
    
    # Adams-Moulton corrector (refine the prediction using u_pred)
    u_next = u_curr + dt * (5/12 * u_dot_pred + 8/12 * u_dot_curr - 1/12 * u_dot_prev)
    
    return u_next


# @jax.jit
def inference_rk(u_curr, trunk_inputs_test, dt):
    
    # Predict the system dynamics (u_dot) at the current state using the model
    u_dot = model_fn(best_params, u_curr, trunk_inputs_test)  # Model's predicted rate of change

    # Implementing the 4th-order Runge-Kutta (RK4) time-stepping method
    k1 = u_dot   #(ns*nt, nx*ny)
    k1 = k1.reshape(k1.shape[0], Nx, Ny)   #(ns*nt, nx, ny)
    
    k2 = model_fn(best_params, u_curr + 0.5 * dt * k1, trunk_inputs_test)   #(ns*nt, nx*ny)
    k2 = k2.reshape(k2.shape[0], Nx, Ny)    #(ns*nt, nx, ny)
    
    k3 = model_fn(best_params, u_curr + 0.5 * dt * k2, trunk_inputs_test)     #(ns*nt, nx*ny)
    k3 = k3.reshape(k3.shape[0], Nx, Ny)    #(ns*nt, nx, ny)
    
    k4 = model_fn(best_params, u_curr + dt * k3, trunk_inputs_test)    #(ns*nt, nx*ny)
    k4 = k4.reshape(k4.shape[0], Nx, Ny)    #(ns*nt, nx, ny)
    
    # Calculate the next state using RK4
    u_pred_next = u_curr + (dt / 6) * (k1 + 2 * k2 + 2 * k3 + k4)  #(ns*nt, nx, ny)
    
    return u_pred_next


def run_inference(initial_u, trunk_inputs_test, n_steps, method, dt):
    u_states = np.zeros(shape = (initial_u.shape[0], n_steps, Nx, Ny))  # List to store the states over time
    u_states[:,0,:,:] = initial_u
    
    # Initialize the previous state (this could be your u_0 and u_1, etc.)
    u_prev = initial_u  # Set the previous state to the initial state
    u_curr = initial_u  # Set the current state to the initial state
    
    for i in range(1, n_steps):
        
        if method == "AB":
            # Perform one inference step using the multistep method
            u_next = inference_ab(u_curr, u_prev, trunk_inputs_test, dt)

            # Append the predicted state to the list
            u_states[:, i, :, :] = u_next

            # Update previous and current states for the next step
            u_prev = u_curr
            u_curr = u_next
        elif method == "RK":
            #Perform one inference step using the rk-4 method
            u_next = inference_rk(u_curr, trunk_inputs_test, dt)
            
            # Append the predicted state to the list
            u_states[:, i, :, :] = u_next
            
            #Update the current state for the next step
            u_curr = u_next
    
    return u_states

In [11]:
Ns, Nt, Nx, Ny = outputs.shape
print(f"Ns: {Ns}, Nt: {Nt}, Nx: {Nx}, Ny: {Ny}")

import time
best_params = load_model_params(path = filepath, filename = filename)
print(f"Best model params loaded with {filename}")

method = "AB"
print(f"Inference scheme being used: {method}")

print("Compute the no. of steps for inference!")
NSTEPS = Nt
print(f"Inference dt: {dt_saved}, nsteps: {NSTEPS}")

Ns: 800, Nt: 101, Nx: 65, Ny: 65
Best model params loaded with model_params_best_21634.pkl
Inference scheme being used: AB
Compute the no. of steps for inference!
Inference dt: [[0.00673435]], nsteps: 101


In [12]:
u_curr = outputs[:, 0, :, :]
start_time = time.time()
u_pred = run_inference(u_curr, trunk_inputs_test, n_steps=NSTEPS, method = method, dt=dt_saved.squeeze())
end_time = time.time()

In [13]:
u_pred.shape, outputs.shape

((800, 101, 65, 65), (800, 101, 65, 65))

In [14]:
u_pred.min(), u_pred.max(), outputs.min(), outputs.max()

(np.float64(-0.4549481272697449),
 np.float64(1.0837446451187134),
 Array(-0.00017021, dtype=float32),
 Array(0.99660045, dtype=float32))

In [15]:
print(f"Total time of inference for {u_pred.shape[0]} samples: ",end_time-start_time)
print(f"Prediction shape: {u_pred.shape}, Outputs shape: {outputs.shape}")

overall_rel_l2_err = jnp.linalg.norm(u_pred - outputs)/jnp.linalg.norm(outputs)
print(f"Overall relative L2 error: {overall_rel_l2_err}")

Total time of inference for 800 samples:  4.885077953338623
Prediction shape: (800, 101, 65, 65), Outputs shape: (800, 101, 65, 65)
Overall relative L2 error: 0.1250378042459488


In [ ]:
#Check the sample for which relative L2 error is minimum
sample_wise_rel_L2_err = []

for i in range(u_pred.shape[0]):
    sample_wise_rel_L2_err.append(
    np.linalg.norm(u_pred[i,:,:,:] - outputs[i,:,:,:])/np.linalg.norm(outputs[i,:,:,:])
    )
sorted_indices = np.argsort(np.array(sample_wise_rel_L2_err))
sorted_indices[:3]

In [ ]:
#Randomly selecting "size" number of samples out of the test dataset
indices = np.random.choice(np.arange(u_pred.shape[0]), size = 3, replace = 'True')

x_test = jnp.linspace(0, 1, Nx)
y_test = jnp.linspace(0, 1, Ny)
t_test = jnp.linspace(0, 1, Nt)

t_query = [0, 25, 50, 75, -1]

for idx in indices:
# for idx in sorted_indices[0:3]:
    n_cols = len(t_query)
    n_rows = 3  # Predicted, Actual, Error
    plt.figure(figsize=(4 * n_cols, 3 * n_rows))

    for j, t in enumerate(t_query):
        # --- Predicted --- #
        plt.subplot(n_rows, n_cols, 1 + j)
        contour1 = plt.contourf(x_test, y_test, u_pred[idx, t, :, :].T, levels=100, cmap='seismic')
        plt.xlabel("x", fontsize=12)
        plt.ylabel("y", fontsize=12)
        plt.xticks(fontsize=10)
        plt.yticks(fontsize=10)
        cbar1 = plt.colorbar()
        cbar1.ax.tick_params(labelsize=10)
        if j == 0:
            plt.ylabel("Predicted", fontsize=14)
        plt.title(f"Timestep {t}", fontsize=14)

        # --- Actual --- #
        plt.subplot(n_rows, n_cols, 1 + n_cols + j)
        contour2 = plt.contourf(x_test, y_test, outputs[idx, t, :, :].T, levels=100, cmap='seismic')
        plt.xlabel("x", fontsize=12)
        plt.ylabel("y", fontsize=12)
        plt.xticks(fontsize=10)
        plt.yticks(fontsize=10)
        cbar2 = plt.colorbar()
        cbar2.ax.tick_params(labelsize=10)
        if j == 0:
            plt.ylabel("Actual", fontsize=14)

        # --- Error --- #
        plt.subplot(n_rows, n_cols, 1 + 2 * n_cols + j)
        contour3 = plt.contourf(x_test, y_test, 
                                jnp.abs(u_pred[idx, t, :, :].T - outputs[idx, t, :, :].T), 
                                cmap='Wistia')
        plt.xlabel("x", fontsize=12)
        plt.ylabel("y", fontsize=12)
        plt.xticks(fontsize=10)
        plt.yticks(fontsize=10)
        cbar3 = plt.colorbar()
        cbar3.ax.tick_params(labelsize=10)
        if j == 0:
            plt.ylabel("Error", fontsize=14)

    plt.suptitle(f"Sample Idx: {idx}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    # plt.savefig(filepath / f"Contour_plots_{idx}_{method}.jpeg", dpi=800)
    plt.show()

In [16]:
#Plotting the relative L2 error obtained at every timestep to show accummulation of autoregressive error
auto_reg_error = []
num_time_steps = Nt

for i in range(num_time_steps):
    l2_error = jnp.linalg.norm(u_pred[:,i,:,:] - outputs[:,i,:,:])/jnp.linalg.norm(outputs[:,i,:,:])
    auto_reg_error.append(l2_error)

#Compute statistics
# t = [10, 20, 30, 40, 50, 60, 70, 90, 100]
t = [52, 64, 88, 100]
print("----Extrapolation errors----")
for t_idx in t:
    print(f"t: {t_idx}, L2 error: {auto_reg_error[t_idx]}")

----Extrapolation errors----
t: 52, L2 error: 0.1093137115240097
t: 64, L2 error: 0.13399729132652283
t: 88, L2 error: 0.19901540875434875
t: 100, L2 error: 0.24013589322566986


In [ ]:
save = True

if save:
    np.savez_compressed(filepath / f"results_{seed}_{method}.npz", 
                        u_pred=u_pred)